# Write a Python Program to implement reverse image search engine using keras

In [2]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
model = VGG16(weights='imagenet', include_top=False, pooling='avg')

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 52s 1us/step


In [ ]:
dataset_path = r"./inputs/caltech-101"

image_paths = []
features_list = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".jpg"):
            path = os.path.join(root, file)
            image_paths.append(path)
            features = extract_features(path)
            features_list.append(features[0])  # append 1D vector, not flattened

features_list = np.array(features_list)  # shape: (num_images, 512)
print("Total images processed:", len(image_paths))
print(features_list.shape)

In [6]:
dataset_path = r"./inputs/caltech-101"

image_paths = []
features_list = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".jpg"):
            path = os.path.join(root, file)
            image_paths.append(path)
            features = extract_features(path)
            features_list = np.append(features_list, features)


features_list = np.array(features_list)
print("Total images processed:", len(image_paths))
print(features_list.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
def search_similar_images(query_img_path, top_k=5):
    """Find top_k most similar images to the query image."""
    query_features = extract_features(query_img_path)  # shape: (1, 512)
    similarities = cosine_similarity(query_features, features_list)  # (1, N)
    similarities = similarities[0]  # flatten to (N,)
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(image_paths[i], similarities[i]) for i in top_indices]

In [ ]:
def display_results(query_img_path, top_k=5):
    """Display query image alongside its top-k similar matches."""
    results = search_similar_images(query_img_path, top_k=top_k)

    fig, axes = plt.subplots(1, top_k + 1, figsize=(3 * (top_k + 1), 3))

    # Show query image
    query_img = cv2.imread(query_img_path)
    query_img = cv2.cvtColor(query_img, cv2.COLOR_BGR2RGB)
    axes[0].imshow(query_img)
    axes[0].set_title("Query", fontsize=10)
    axes[0].axis("off")

    # Show similar images
    for i, (img_path, score) in enumerate(results):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i + 1].imshow(img)
        axes[i + 1].set_title(f"Score: {score:.2f}", fontsize=9)
        axes[i + 1].axis("off")

    plt.suptitle("Reverse Image Search Results", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Example usage — replace with any .jpg path from your dataset
if image_paths:
    query_image = image_paths[0]
    display_results(query_image, top_k=5)
else:
    print("No images found. Check your dataset_path.")